# 1. Importation des bibliothèques et des jeux de données
Importer pandas, scikit-learn et charger les fichiers 'features.parquet' et 'target.parquet' depuis le répertoire 'data/modelling/'.

In [4]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Chemins des données
DATA_DIR = Path("data/modelling")
FEATURES_PATH = DATA_DIR / "features.parquet"
TARGET_PATH = DATA_DIR / "target.parquet"

# Chargement des données
df_features = pq.read_table(FEATURES_PATH).to_pandas()
df_target = pq.read_table(TARGET_PATH).to_pandas()

print(f"Features shape: {df_features.shape}")
print(f"Target shape: {df_target.shape}")

Features shape: (17952768, 6)
Target shape: (17952768, 1)


# 2. Fusion des ensembles de données (Target et Features)
Utiliser pandas pour fusionner les deux DataFrames sur la clé commune (individual et timestamp).

In [5]:
# Fusion sur l'index (individual, timestamp)
df_full = df_features.join(df_target, how="inner")

print(f"Full dataset shape: {df_full.shape}")
df_full.head()

Full dataset shape: (17952768, 7)


lag_1d  lag_7d  lag_30d  lag_365d  \
individual timestamp                                                
MT_124     2011-01-01 00:15:00     NaN     NaN      NaN       NaN   
           2011-01-01 00:30:00     NaN     NaN      NaN       NaN   
           2011-01-01 00:45:00     NaN     NaN      NaN       NaN   
           2011-01-01 01:00:00     NaN     NaN      NaN       NaN   
           2011-01-01 01:15:00     NaN     NaN      NaN       NaN   

                                rolling_mean_7d  rolling_mean_30d  \
individual timestamp                                                
MT_124     2011-01-01 00:15:00              NaN               NaN   
           2011-01-01 00:30:00              NaN               NaN   
           2011-01-01 00:45:00              NaN               NaN   
           2011-01-01 01:00:00              NaN               NaN   
           2011-01-01 01:15:00              NaN               NaN   

                                consumption_kwh  
individual timestamp                             
MT_124     2011-01-01 00:15:00        17.942584  
           2011-01-01 00:30:00        17.942584  
           2011-01-01 00:45:00        15.550239  
           2011-01-01 01:00:00        17.942584  
           2011-01-01 01:15:00        16.746411

# 3. Prétraitement et nettoyage des données
Gérer les valeurs manquantes et séparer les variables explicatives (X) de la cible (y).

In [6]:
# Suppression des lignes avec des valeurs manquantes ( dues aux lags et rolling means )
df_clean = df_full.dropna()

X = df_clean.drop(columns=["consumption_kwh"])
y = df_clean["consumption_kwh"]

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (13467648, 6), y shape: (13467648,)


# 4. Séparation en ensembles d'entraînement et de test
Utiliser train_test_split pour séparer les données avec un ratio typique de 80/20.

In [7]:
# Séparation train/test par années
train_years = [2011, 2012]
test_years = [2013, 2014]

timestamp_years = df_clean.index.get_level_values("timestamp").year
train_mask = np.isin(timestamp_years, train_years)
test_mask = np.isin(timestamp_years, test_years)

df_train = df_clean.loc[train_mask]
df_test = df_clean.loc[test_mask]

X_train = df_train.drop(columns=["consumption_kwh"])
y_train = df_train["consumption_kwh"]
X_test = df_test.drop(columns=["consumption_kwh"])
y_test = df_test["consumption_kwh"]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (4497280, 6), Test shape: (8970240, 6)


# 5. Entraînement du modèle de machine learning
Initialiser un estimateur (ex: RandomForestRegressor) et l'entraîner via la méthode fit().

In [ ]:
# Entraînement du modèle
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# Prédictions et évaluation
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")